In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/playground-series-s6e6/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e6/train.csv
/kaggle/input/competitions/playground-series-s6e6/test.csv


In [2]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

# 1. データの読み込み
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e6/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e6/test.csv')

target_col = 'class'

# ラベルエンコーディング (GALAXY, STAR, QSO -> 0, 1, 2)
le = LabelEncoder()
X = train.drop(columns=['id', target_col], errors='ignore')
y = le.fit_transform(train[target_col])
X_test = test.drop(columns=['id'], errors='ignore')

# カテゴリ変数の型変換
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
for col in cat_cols:
    X[col] = X[col].astype('category')
    X_test[col] = X_test[col].astype('category')

num_classes = len(le.classes_)

# 2. 交差検証の設計（5-Fold）
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# それぞれのモデルの予測確率を格納する配列を初期化
oof_preds_xgb = np.zeros((len(X), num_classes))
oof_preds_lgb = np.zeros((len(X), num_classes))

test_preds_xgb = np.zeros((len(X_test), num_classes))
test_preds_lgb = np.zeros((len(X_test), num_classes))

# 3. 学習ループ
import os
import sys

# 3. 学習ループ
for fold, (train_idx, val_idx) in enumerate(folds.split(X, y)):
    print(f"\n==================== FOLD {fold+1} ====================")
    X_train, y_train = X.iloc[train_idx], y[train_idx]
    X_val, y_val = X.iloc[val_idx], y[val_idx]
    
    # --- モデル①: XGBoost (GPUモード) ---
    print("Training XGBoost...")
    model_xgb = xgb.XGBClassifier(
        n_estimators=1000,
        max_depth=6,
        learning_rate=0.05,
        objective='multi:softprob',
        num_class=num_classes,
        tree_method='hist',
        device='cuda',
        enable_categorical=True,
        early_stopping_rounds=30,
        random_state=42
    )
    model_xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    
    oof_preds_xgb[val_idx] = model_xgb.predict_proba(X_val)
    test_preds_xgb += model_xgb.predict_proba(X_test) / folds.n_splits
    
    # --- モデル②: LightGBM (警告を物理的にシャットアウト) ---
    print("Training LightGBM...")
    model_lgb = lgb.LGBMClassifier(
        n_estimators=1000,
        max_depth=6,
        learning_rate=0.05,
        objective='multiclass',
        num_class=num_classes,
        device='gpu',
        random_state=42,
        verbose=-1
    )
    
    # 【数理エンジニアの裏技】C++レベルの警告ログ連打をシステム的に完全にブロックする
    null_fds = [os.open(os.devnull, os.O_RDWR) for _ in range(2)]
    save_fds = [os.dup(1), os.dup(2)]
    os.dup2(null_fds[0], 1)
    os.dup2(null_fds[1], 2)
    
    try:
        # この中で起きる警告は一切画面に表示されなくなります
        model_lgb.fit(
            X_train, y_train, 
            eval_set=[(X_val, y_val)], 
            callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)]
        )
    finally:
        # 学習が終わったらログ出力を元に戻す
        os.dup2(save_fds[0], 1)
        os.dup2(save_fds[1], 2)
        os.close(null_fds[0])
        os.close(null_fds[1])
        os.close(save_fds[0])
        os.close(save_fds[1])
    
    oof_preds_lgb[val_idx] = model_lgb.predict_proba(X_val)
    test_preds_lgb += model_lgb.predict_proba(X_test) / folds.n_splits
    print(f"FOLD {fold+1} Finished!")
# 4. 単体モデルのスコア確認
score_xgb = accuracy_score(y, np.argmax(oof_preds_xgb, axis=1))
score_lgb = accuracy_score(y, np.argmax(oof_preds_lgb, axis=1))
print(f"\n[単体スコア] XGBoost Accuracy: {score_xgb:.5f}")
print(f"[単体スコア] LightGBM Accuracy: {score_lgb:.5f}")

# 5. 数理的最適化：2つのモデルを 50% ずつブレンド（平均化）
print("\nBlending predictions (50% XGBoost + 50% LightGBM)...")
oof_preds_blend = (oof_preds_xgb * 0.5) + (oof_preds_lgb * 0.5)
test_preds_blend = (test_preds_xgb * 0.5) + (test_preds_lgb * 0.5)

# ブレンド後の最終スコア
score_blend = accuracy_score(y, np.argmax(oof_preds_blend, axis=1))
print(f"==================================================")
print(f"🔥 ブレンド後（OOF）の予測正解率 (Accuracy): {score_blend:.5f}")
print(f"==================================================")

# 6. 提出用ファイルの作成
sub = pd.DataFrame({'id': test['id']})
sub[target_col] = le.inverse_transform(np.argmax(test_preds_blend, axis=1))
sub.to_csv("submission.csv", index=False)
print("提出用ファイル 'submission.csv' が正常に更新されました！")


==================== FOLD 1 ====================
Training XGBoost...
Training LightGBM...
FOLD 1 Finished!

==================== FOLD 2 ====================
Training XGBoost...
Training LightGBM...
FOLD 2 Finished!

==================== FOLD 3 ====================
Training XGBoost...
Training LightGBM...
FOLD 3 Finished!

==================== FOLD 4 ====================
Training XGBoost...
Training LightGBM...
FOLD 4 Finished!

==================== FOLD 5 ====================
Training XGBoost...
Training LightGBM...
FOLD 5 Finished!

[単体スコア] XGBoost Accuracy: 0.96701
[単体スコア] LightGBM Accuracy: 0.96745

Blending predictions (50% XGBoost + 50% LightGBM)...
🔥 ブレンド後（OOF）の予測正解率 (Accuracy): 0.96775
提出用ファイル 'submission.csv' が正常に更新されました！
